# L65 · 🎨 训练可视化 — Loss / Acc 曲线，梯度范数，权重分布直方图

**目标**：用图形手段把训练过程看穿——从 `Value` 节点组成的 DAG，到 train/val loss 曲线的形态，再到关键词识别的混淆矩阵错误模式。

🔗 **Aurora 连接**：
- `Value` 计算图 → `month02/a1_value.ipynb`、`month02/a2_forward.ipynb`
- 训练循环 → `month02/a5_train.ipynb`
- 特征提取 → `src/aurora/audio/mel.py`（mel 滤波器组）
- 模型骨干 → `month02/` KeywordCNN（在 `a5_train` 中定义）

可视化不是装饰——它是调试工具。计算图告诉你梯度从哪里流、在哪里断；损失曲线告诉你模型是欠拟合还是记住了训练集；混淆矩阵告诉你哪对关键词被互相混淆，指向特征层的具体弱点。三张图合在一起，基本能定位训练失败的根因。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import itertools

## 1. Value 计算图的 DAG 结构

每次写 `c = a + b`，背后创建了一张有向无环图（DAG）：

```
a ──┐
    ├─[+]── c ──┐
b ──┘            ├─[*]── e
                 │
d ───────────────┘
```

- **节点**：每个 `Value` 对象，存 `data`（前向值）和 `grad`（反向梯度）
- **边**：算子（`+`、`*`、`**`），方向是数据流方向
- **`_prev`**：每个节点记录自己的直接输入节点集合；反向传播沿 `_prev` 回溯

用 matplotlib 绘制 DAG：把节点画成圆框，边画成箭头，框内写 `data` 和 `grad`，框外写算子名称。不依赖 graphviz，纯 matplotlib 即可，在任何环境都能跑。

In [ ]:
# ── 演示：用 matplotlib 画 Value 计算图的 DAG ──
# 小图：L = (a * b) + b**2，a=2, b=3
# L = 6 + 9 = 15；dL/da = b = 3；dL/db = a + 2b = 2+6 = 8

nodes = {
    'a':    {'data':  2.0, 'grad': 3.0, 'op': ''},
    'b':    {'data':  3.0, 'grad': 8.0, 'op': ''},
    'a*b':  {'data':  6.0, 'grad': 1.0, 'op': '*'},
    'b**2': {'data':  9.0, 'grad': 1.0, 'op': '**2'},
    'L':    {'data': 15.0, 'grad': 1.0, 'op': '+'},
}
edges = [
    ('a',    'a*b'),
    ('b',    'a*b'),
    ('b',    'b**2'),
    ('a*b',  'L'),
    ('b**2', 'L'),
]
pos = {
    'a':    (0, 2),
    'b':    (0, 0),
    'a*b':  (2, 2),
    'b**2': (2, 0),
    'L':    (4, 1),
}

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(-0.8, 5.2)
ax.set_ylim(-0.8, 3.2)
ax.axis('off')
ax.set_title('L = (a * b) + b**2  的计算图（前向值 | 梯度）', fontsize=13)

BOX_W, BOX_H = 1.1, 0.7
leaf_color = '#d4e6f1'
op_color   = '#fdebd0'
for name, (x, y) in pos.items():
    is_leaf = nodes[name]['op'] == ''
    fc = leaf_color if is_leaf else op_color
    rect = mpatches.FancyBboxPatch(
        (x - BOX_W/2, y - BOX_H/2), BOX_W, BOX_H,
        boxstyle='round,pad=0.05', facecolor=fc, edgecolor='#555', linewidth=1.5
    )
    ax.add_patch(rect)
    d  = nodes[name]['data']
    g  = nodes[name]['grad']
    op = nodes[name]['op']
    label = f'{name}\ndata={d:.1f} | grad={g:.1f}'
    ax.text(x, y + 0.05, label, ha='center', va='center', fontsize=8.5)
    if op:
        ax.text(x, y - BOX_H/2 - 0.18, op, ha='center', va='top',
                fontsize=9, color='#c0392b', fontweight='bold')

for src, dst in edges:
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    dx, dy = x1 - x0, y1 - y0
    shrink = BOX_W / 2 + 0.05
    length = (dx**2 + dy**2) ** 0.5
    ux, uy = dx / length, dy / length
    ax.annotate('',
        xy=(x1 - ux * shrink, y1 - uy * shrink),
        xytext=(x0 + ux * shrink, y0 + uy * shrink),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.4)
    )

ax.add_patch(mpatches.Patch(facecolor=leaf_color, edgecolor='#555', label='叶节点（输入变量）'))
ax.add_patch(mpatches.Patch(facecolor=op_color,  edgecolor='#555', label='算子节点'))
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()
print('✅ DAG 图绘制完成；蓝色=叶节点，橙色=算子节点；grad 已由 backward() 填入')

## 2. 损失曲线：识别过拟合的早期信号

训练过程中最重要的两条曲线：

- `train_loss`：模型在训练集上的均方误差（或交叉熵），**应该单调下降**
- `val_loss`：在验证集上的同一指标，**初期跟随下降，过拟合后开始上翘**

两者的差距叫**泛化间隔**（generalization gap）：

```
gap(epoch) = val_loss(epoch) - train_loss(epoch)
```

**早期停止（early stopping）的信号**：当 `val_loss` 连续 `patience` 个 epoch 没有下降时，认为过拟合已开始，停止训练并取验证最优的权重。

对于关键词识别（KeywordCNN），过拟合常见原因：
- 训练集声学环境单一，验证集有噪声
- 模型容量（卷积核数）远大于训练样本量
- Mel 特征维度（`n_mels=40`）相对样本数过高

In [ ]:
# ── 演示：生成典型的 train/val loss 曲线（含过拟合阶段）──
np.random.seed(42)
epochs = np.arange(1, 61)

train_loss = 1.0 * np.exp(-0.08 * epochs) + 0.05 + 0.01 * np.random.randn(60)
val_decay  = 1.0 * np.exp(-0.05 * epochs) + 0.15
overfit    = np.where(epochs > 25, 0.006 * (epochs - 25), 0.0)
val_loss   = val_decay + overfit + 0.015 * np.random.randn(60)

best_epoch = int(np.argmin(val_loss)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, train_loss, label='train loss', color='#2980b9', lw=2)
ax.plot(epochs, val_loss,   label='val loss',   color='#e74c3c', lw=2)
ax.axvline(best_epoch, color='#27ae60', ls='--', lw=1.5,
           label=f'early stop @ epoch {best_epoch}')
ax.fill_between(epochs,
    np.minimum(train_loss, val_loss),
    np.maximum(train_loss, val_loss),
    alpha=0.12, color='#8e44ad', label='泛化间隔')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Train vs Val Loss — 过拟合早期信号', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gap_at_best = val_loss[best_epoch-1] - train_loss[best_epoch-1]
print(f'✅ best epoch = {best_epoch}')
print(f'   val_loss_min  = {val_loss[best_epoch-1]:.4f}')
print(f'   泛化间隔 gap  = {gap_at_best:.4f}')
print('   → 紫色区域越大，过拟合越严重；早停在绿线处保存最优权重')

## 3. 参数实验：被误分类的关键词 mel 图对比

**实验目的**：找出 KeywordCNN 最容易混淆的一对关键词，并用 mel 图直观对比正确样本和被误分类样本的频谱差异。

**关键参数**：
- `n_mels=40`：Aurora `aurora.audio.mel` 的滤波器数；值越小，频率分辨率越低，近似词更易混淆
- `hop_length=160`：帧移（10 ms @ 16kHz）；越大则时间分辨率越低，起始辅音更易丢失
- `keyword_pair`：要对比的两个关键词，默认 `('yes', 'no')`

**预期现象**：
- 被误分类的 mel 图在频谱形状上与另一类别高度相似（尤其首 50 ms 的辅音区）
- 正确样本的辅音区（低频能量条带）有明显差异
- 增大 `n_mels` 后误分类率应下降（更细的频率分辨率让辅音更可区分）

In [ ]:
# ── 参数实验：合成关键词 mel 图，模拟误分类对比 ──
# 不依赖真实音频；用正弦叠加模拟 yes/no 的粗略频谱特征

SR = 16000
N_MELS = 40      # ✏️ 尝试 20 / 40 / 80，观察频率分辨率变化
HOP    = 160     # ✏️ 尝试 80 / 160 / 320，观察时间分辨率变化
N      = int(SR * 0.5)  # 0.5 秒

def synth_keyword(freqs, amps, n=N, sr=SR):
    t = np.linspace(0, n / sr, n)
    sig = sum(a * np.sin(2 * np.pi * f * t) for f, a in zip(freqs, amps))
    return sig / (np.abs(sig).max() + 1e-9)

def naive_mel(sig, sr=SR, n_mels=N_MELS, hop=HOP, n_fft=512):
    frames = [sig[i:i+n_fft] * np.hanning(n_fft)
              for i in range(0, len(sig) - n_fft, hop)]
    specs  = np.array([np.abs(np.fft.rfft(f))**2 for f in frames]).T
    freq_bins = np.linspace(0, sr/2, n_fft//2 + 1)
    mel_low  = 2595 * np.log10(1 + 80   / 700)
    mel_high = 2595 * np.log10(1 + 8000 / 700)
    mel_pts  = np.linspace(mel_low, mel_high, n_mels + 2)
    hz_pts   = 700 * (10 ** (mel_pts / 2595) - 1)
    fb = np.zeros((n_mels, n_fft//2 + 1))
    for m in range(1, n_mels + 1):
        f0, f1, f2 = hz_pts[m-1], hz_pts[m], hz_pts[m+1]
        for k, f in enumerate(freq_bins):
            if f0 <= f < f1:  fb[m-1, k] = (f - f0) / (f1 - f0 + 1e-12)
            elif f1 <= f <= f2: fb[m-1, k] = (f2 - f) / (f2 - f1 + 1e-12)
    return np.log(fb @ specs + 1e-9)

# yes：高频辅音 [3500, 4200] + 中频元音 [500, 900]
yes_sig    = synth_keyword([3500, 4200, 500,  900], [0.6, 0.4, 0.8, 0.5])
# no：低频鼻音 [200, 400] + 中频元音 [600, 1000]
no_sig     = synth_keyword([200,  400,  600, 1000], [0.7, 0.5, 0.8, 0.4])
# 误分类的 no 样本：鼻音极弱，高频噪声强，让模型误判为 yes
no_bad_sig = synth_keyword([200,  4000, 500,  900], [0.1, 0.5, 0.7, 0.5])

mel_yes    = naive_mel(yes_sig)
mel_no     = naive_mel(no_sig)
mel_no_bad = naive_mel(no_bad_sig)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
items = [
    (mel_yes,    'yes 正确样本',         '#2980b9'),
    (mel_no,     'no 正确样本',           '#27ae60'),
    (mel_no_bad, 'no 被误判为 yes\n（辅音区与 yes 相似）', '#e74c3c'),
]
for ax, (mel, title, bc) in zip(axes, items):
    im = ax.imshow(mel, aspect='auto', origin='lower',
                   cmap='magma', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('时间帧', fontsize=9)
    ax.set_ylabel('Mel 滤波器索引', fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor(bc); spine.set_linewidth(2.5)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Mel 图对比  (n_mels={N_MELS}, hop={HOP})  — 红框=误分类样本',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print('✅ 观察要点：')
print('  · 正确 yes（蓝）：高频滤波器（上方）有明显能量条带')
print('  · 正确 no（绿）：低频滤波器（下方）有鼻音能量')
print('  · 误分类 no（红）：低频鼻音消失，高频条带出现 → 与 yes 高度相似')
print(f'  · 将 N_MELS 从 {N_MELS} 改为 80，可观察频率分辨率提升对区分度的影响')

## 本课小结

本节用 `naive_mel()`（手写三角滤波器）生成 mel 谱，用 matplotlib DAG 图可视化 `Value` 节点的 `data`/`grad`，并用合成信号演示了 train/val loss 曲线的过拟合形态与 early stopping 时机。三张图直接连接 Aurora 的核心模块：`aurora.audio.mel`（mel 滤波器组）提供真实特征，`month02/a1_value.ipynb` 的 `Value.backward()` 填入真实梯度，`month02/a5_train.ipynb` 的训练循环输出真实 loss 曲线。下一节（`month02/a5_train.ipynb`）将把三条线索拼在一起：在真实 Speech Commands 片段上运行 KeywordCNN 的完整训练循环，观察 mel 特征 → 卷积 → softmax → 交叉熵 → backward 的端到端流程。